# 03 - Red neuronal con TensorFlow / Keras

Clasificación binaria: **0 = normal, 1 = posible fraude**. Se utiliza StandardScaler, pesos de clase, Early Stopping y ReduceLROnPlateau.


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

BASE_DIR = Path.cwd().parent
df = pd.read_csv(BASE_DIR / "datos" / "dataset_fraude_yape.csv")
df["fecha_hora"] = pd.to_datetime(df["fecha_hora"])
df["hora"] = df["fecha_hora"].dt.hour
df["dia_semana"] = df["fecha_hora"].dt.dayofweek

X = df.drop(columns=["id_transaccion", "fecha_hora", "puntaje_riesgo", "nivel_riesgo", "fraude"])
y = df["fraude"].astype(int)
X = pd.get_dummies(X, columns=["producto"], dtype=float).fillna(0).astype(float)

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

normal = int((y_train == 0).sum())
fraude = int((y_train == 1).sum())
class_weight = {0: 1.0, 1: normal / fraude}
print("Variables:", X_train.shape[1])
print("Pesos:", class_weight)


In [ ]:
modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

modelo.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
               loss="binary_crossentropy", metrics=["accuracy"])
modelo.summary()


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)
]

history = modelo.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=50, batch_size=256, class_weight=class_weight,
    callbacks=callbacks, verbose=1)


In [ ]:
prob = modelo.predict(X_test, verbose=0).ravel()
pred = (prob >= 0.5).astype(int)

metricas = {
    "Accuracy": accuracy_score(y_test, pred),
    "Precision": precision_score(y_test, pred, zero_division=0),
    "Recall": recall_score(y_test, pred, zero_division=0),
    "F1": f1_score(y_test, pred, zero_division=0)
}
print(metricas)
print("\nMatriz de confusión:\n", confusion_matrix(y_test, pred))
print("\n", classification_report(y_test, pred, target_names=["Normal", "Fraude"]))


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history["loss"], label="Entrenamiento")
plt.plot(history.history["val_loss"], label="Validación")
plt.title("Pérdida durante el entrenamiento")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
MODELOS_DIR = BASE_DIR / "modelos"
MODELOS_DIR.mkdir(exist_ok=True)
modelo.save(MODELOS_DIR / "modelo_tensorflow.keras")
print("Modelo TensorFlow guardado.")


## Resultado de referencia

TensorFlow/Keras obtuvo aproximadamente **95.27% Accuracy, 58.07% Precision, 95.50% Recall y 72.23% F1** en la ejecución del proyecto.
